# note

I FORGOT TO DO TRAIN TEST EVAL BRUH see v2 ig

# step 1: setup and imports

In [1]:
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from transformers import BertTokenizer, BertForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import f1_score, roc_auc_score
from datasets import Dataset

# Emotion label columns — all 28 of them
EMOTION_COLS = [
    'admiration', 'amusement', 'anger', 'annoyance', 'approval', 'caring',
    'confusion', 'curiosity', 'desire', 'disappointment', 'disapproval',
    'disgust', 'embarrassment', 'excitement', 'fear', 'gratitude', 'grief',
    'joy', 'love', 'nervousness', 'optimism', 'pride', 'realization',
    'relief', 'remorse', 'sadness', 'surprise', 'neutral'
]

/Users/zacchaeus_tan/Documents/smu/dap/zac-personal-learning-transformers/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Step 2 — Load & clean your data

In [3]:
df = pd.read_csv("../goemotions_cleaned.csv")
df = df[['text'] + EMOTION_COLS]
df = df[df[EMOTION_COLS].sum(axis=1) > 0].reset_index(drop=True)



# Step 3 — Tokenize


In [4]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

def tokenize(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=128  # Reddit comments are usually short
    )

# Step 4 — Build the dataset


In [5]:
# Step 1: split off test (10%) from everything else (90%)
train_val_df, test_df = train_test_split(df, test_size=0.1, random_state=42)

# Step 2: split the 90% into train (80%) and eval (20%)
# This gives you roughly 80% train / 10% eval / 10% test overall
train_df, eval_df = train_test_split(train_val_df, test_size=0.11, random_state=42)

print(f"Train:      {len(train_df):,} rows")
print(f"Eval:       {len(eval_df):,} rows")
print(f"Test:       {len(test_df):,} rows")
print(f"Total:      {len(df):,} rows")

train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True))
eval_dataset  = Dataset.from_pandas(eval_df.reset_index(drop=True))
test_dataset  = Dataset.from_pandas(test_df.reset_index(drop=True))

# Tokenize all three
train_dataset = train_dataset.map(tokenize, batched=True)
eval_dataset  = eval_dataset.map(tokenize, batched=True)
test_dataset  = test_dataset.map(tokenize, batched=True)


# Tell the dataset which columns to use during training
# ⚠️ For multi-label, labels must be a float tensor, not int
def format_labels(example):
    example["labels"] = [float(example[col]) for col in EMOTION_COLS]
    return example

train_dataset = train_dataset.map(format_labels)
eval_dataset  = eval_dataset.map(format_labels)
test_dataset  = test_dataset.map(format_labels)

train_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
eval_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
test_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

Train:      45,782 rows
Eval:       5,659 rows
Test:       5,716 rows
Total:      57,157 rows


Map: 100%|██████████| 5716/5716 [00:00<00:00, 10513.98 examples/s]


# Step 5 — Load the v1 model 


In [ ]:
model = BertForSequenceClassification.from_pretrained("../BERTv1/my_goemotions_modelv1")
tokenizer = BertTokenizer.from_pretrained("../BERTv1/my_goemotions_modelv1")

# Check 1 — confirm it's the right architecture and number of labels. run the predict_emotions cell from below first
print("Number of labels:", model.config.num_labels)   # should print 28
print("Model type:", model.config.model_type)          # should print 'bert'

# Check 2 — make sure it's ready for inference
model.eval()
print("Model loaded and ready!")

# Check 3 — run a quick test prediction
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print(predict_emotions("I am so happy today!"))   # should return joy/excitement etc.

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 2707.39it/s, Materializing param=classifier.weight]                                      

Number of labels: 28
Model type: bert
Model loaded and ready!
{'joy': 0.574}


# Step 6 — Evaluation metrics


In [10]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    # Convert logits to predictions using 0.5 threshold
    predictions = (torch.sigmoid(torch.tensor(logits)) > 0.5).int().numpy()
    labels = labels.astype(int)
    return {
        "f1_micro": f1_score(labels, predictions, average="micro", zero_division=0),
        "f1_macro": f1_score(labels, predictions, average="macro", zero_division=0),
    }

# Step 7 — Train


In [11]:
%pip install "accelerate>=1.1.0" -q


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [12]:
import torch
print(torch.cuda.is_available())

False


In [13]:
training_args = TrainingArguments(
    output_dir="./goemotions_bert",
    num_train_epochs=3,
    per_device_train_batch_size=16,   # lower to 8 if you get out-of-memory errors
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_micro",
    logging_steps=100,
    warmup_steps=200,                 # helps BERT warm up gradually at the start
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,     # ← used during training to monitor performance
    compute_metrics=compute_metrics,
)

trainer.train()

/Users/zacchaeus_tan/Documents/smu/dap/zac-personal-learning-transformers/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

- The only thing you need to change between runs is the run_name variable at the top — just update it to something descriptive like "run2_epochs5_lr2e5" before each new training run. The comparison section at the bottom will automatically pick up all saved CSVs and print them side by side.

In [ ]:
import pandas as pd

# Extract and display validation history
log_history = trainer.state.log_history
eval_logs = [log for log in log_history if 'eval_f1_micro' in log]
results_df = pd.DataFrame(eval_logs)[['epoch', 'eval_loss', 'eval_f1_micro', 'eval_f1_macro']]
print(results_df)

# Save to CSV with descriptive name — change this name each run!
run_name = "run1_epochs3_lr2e5"
results_df.to_csv(f"{run_name}.csv", index=False)
print(f"\nSaved to {run_name}.csv")

# Compare all saved runs (will grow as you do more runs)
import glob
all_runs = glob.glob("run*.csv")
for f in sorted(all_runs):
    df = pd.read_csv(f)
    print(f"{f:30s} | best F1 Micro: {df['eval_f1_micro'].max():.4f} | best F1 Macro: {df['eval_f1_macro'].max():.4f}")

# Step 8 - test with the test dataset

- only proceed when we're ready to test

### won't training again with different number of epochs be bad for the model since the model has already 'seen' the test set?

- yes:  if you look at your test score, tweak something (epochs, learning rate, etc.), retrain, and check test again repeatedly. Each time you peek at the test score and make a decision based on it, you're indirectly "leaking" information from the test set into your choices.

- no: The model never trained on it — it was only used in trainer.evaluate(test_dataset) for read-only scoring, not for updating the model's weights. So it's still a valid final benchmark.

- best practice
1. train 
2. check EVAL score 
3. adjust hyperparameters (only use this to make decisions)
4. retrain

a. final test score -> run ONCE at the very end when you're done experimenting

In [ ]:
# This is your final, honest performance score
results = trainer.evaluate(test_dataset)
print("Final Test Results:")
print(f"  F1 Micro: {results['eval_f1_micro']:.4f}")
print(f"  F1 Macro: {results['eval_f1_macro']:.4f}")

# Step 9 — Make predictions on new text


In [8]:
def predict_emotions(text, threshold=0.5):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=128)
    with torch.no_grad():
        outputs = model(**inputs)
    probs = torch.sigmoid(outputs.logits[0])
    results = {EMOTION_COLS[i]: round(probs[i].item(), 3) for i in range(28)}
    # Only show emotions above the threshold
    active = {k: v for k, v in results.items() if v >= threshold}
    return dict(sorted(active.items(), key=lambda x: -x[1]))

print(predict_emotions("I can't believe how amazing that was!"))
# → {'admiration': 0.82, 'excitement': 0.71, 'joy': 0.63}

print(predict_emotions("This makes me so angry and disappointed."))
# → {'anger': 0.79, 'disappointment': 0.68, 'annoyance': 0.54}

{}
{}


# Step 10 — Save


- to save multiple versions just change the folder name

- model.save_pretrained("./goemotions_epochs3")   
- model.save_pretrained("./goemotions_epochs5")   

In [ ]:
model.save_pretrained("./my_goemotions_model3")
tokenizer.save_pretrained("./my_goemotions_model3")